In [1]:
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn

In [3]:
R = sp.load_npz("data/netflix_ratings_csr.npz")
n_movies, n_users = R.shape
n_movies, n_users

(17770, 480189)

In [4]:
coo = R.tocoo()
m = coo.row.astype(np.int32)                 
u = coo.col.astype(np.int32)
r = coo.data.astype(np.float32)              
m,u,r

(array([    0,     0,     0, ..., 17769, 17769, 17769],
       shape=(100480507,), dtype=int32),
 array([   163,    433,    578, ..., 478755, 479027, 479767],
       shape=(100480507,), dtype=int32),
 array([5., 3., 3., ..., 3., 3., 2.], shape=(100480507,), dtype=float32))

In [8]:
rng = np.random.default_rng(42)
mask = rng.random(len(r)) < 0.95
mask

array([ True,  True,  True, ...,  True, False,  True], shape=(100480507,))

In [7]:
m_tr, u_tr, r_tr = m[mask], u[mask], r[mask]
m_va, u_va, r_va = m[~mask], u[~mask], r[~mask]
print(n_movies, n_users)
print(len(r_tr), len(r_va))
print(r_tr.mean())

17770 480189
95455695 5024812
3.6042552


In [9]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)

device: cpu


In [10]:
class MF(nn.Module):
    def __init__(self, n_movies, n_user, k, mu):
        super().__init__()
        self.P  = nn.Embedding(n_movies, k)   # movie factors
        self.Q  = nn.Embedding(n_users, k)    # user factors
        self.bi = nn.Embedding(n_movies, 1)   # movie bias
        self.bu = nn.Embedding(n_users, 1)    # user bias
        self.register_buffer("mu", torch.tensor(mu))   # global mean

        nn.init.normal_(self.P.weight, std=0.01)
        nn.init.normal_(self.Q.weight, std=0.01)
        nn.init.zeros_(self.bi.weight)
        nn.init.zeros_(self.bu.weight)

    def forward(self, i, u):
        p, q = self.P(i), self.Q(u)
        pred = self.mu + self.bi(i).squeeze(1) + self.bu(u).squeeze(1) + (p * q).sum(1)
        return pred, p, q      # p, q are returned so we can regularize them later    

In [11]:
k = 64
model = MF(n_movies, n_users, k, mu=float(r_tr.mean())).to(dev)
print(sum(p.numel() for p in model.parameters()))

32367335


In [12]:
i = torch.tensor(m_tr[:5], dtype=torch.long, device=dev)
uu = torch.tensor(u_tr[:5], dtype=torch.long, device=dev)
print(model(i, uu)[0])      # five numbers all very close to 3.6

tensor([3.6032, 3.6041, 3.6041, 3.6051, 3.6035], grad_fn=<AddBackward0>)


In [13]:
@torch.no_grad()
def val_rmse(chunk=1_000_000):
    model.eval()
    se = 0.0
    for s in range(0, len(r_va), chunk):
        i  = torch.from_numpy(m_va[s:s+chunk]).long().to(dev)
        uu = torch.from_numpy(u_va[s:s+chunk]).long().to(dev)
        r  = torch.from_numpy(r_va[s:s+chunk]).to(dev)
        pred, _, _ = model(i, uu)
        se += (pred.clamp(1, 5) - r).pow(2).sum().item()
    model.train()
    return (se / len(r_va)) ** 0.5

print("RMSE before training:", val_rmse())   # expect about 1.12 (std of ratings)

RMSE before training: 1.084914460724558


In [14]:
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
lam, lam_b, bs = 0.05, 0.005, 65_536

# test slice: first 5M training ratings (random order already, since the split was random)
N = 5_000_000
I = torch.from_numpy(m_tr[:N]).long()
U = torch.from_numpy(u_tr[:N]).long()
Rt = torch.from_numpy(r_tr[:N])

perm = torch.randperm(N)
for step, s in enumerate(range(0, N, bs)):
    idx = perm[s:s+bs]
    i, uu, r = I[idx].to(dev), U[idx].to(dev), Rt[idx].to(dev)

    pred, p, q = model(i, uu)
    mse = (pred - r).pow(2).mean()
    reg = lam * (p.pow(2).sum(1) + q.pow(2).sum(1)).mean() \
        + lam_b * (model.bi(i).pow(2) + model.bu(uu).pow(2)).mean()

    opt.zero_grad()
    (mse + reg).backward()
    opt.step()

    if step % 15 == 0:
        print(step, round(mse.item(), 4))

print("RMSE after test slice:", val_rmse())

0 1.1511
15 1.1017
30 1.0501
45 0.978
60 0.9361
75 0.9127
RMSE after test slice: 1.058781465345164
